# 09 — Analyse comparative des résultats

Ce notebook consolide les résultats A/B/C/D produits dans la matrice comparative. Il distingue les performances d'exécution BIRD des évaluations secondaires de retrieval documentaire.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MATRIX = ROOT / 'results/figures/comparative_matrix.json'
with MATRIX.open(encoding='utf-8') as stream:
    matrix = json.load(stream)
rows = matrix.get('pipelines', matrix)
df = pd.DataFrame(rows).T if isinstance(rows, dict) else pd.DataFrame(rows)
df

In [ ]:
metric = 'execution_accuracy'
if metric in df.columns:
    ax = (100 * pd.to_numeric(df[metric], errors='coerce')).plot.bar(color=['#64748b', '#2563eb', '#a855f7', '#16a34a'], figsize=(7, 4))
    ax.set(title='Execution Accuracy par pipeline', xlabel='Pipeline', ylabel='Execution Accuracy (%)')
    plt.xticks(rotation=0); plt.tight_layout()
else:
    print('Colonne execution_accuracy absente : inspectez la matrice ci-dessus.')

In [ ]:
error_metrics = [name for name in ['valid_sql_rate', 'refusal_rate', 'static_structural_error_rate', 'execution_structural_error_rate', 'wrong_execution_result_rate'] if name in df.columns]
display((100 * df[error_metrics].apply(pd.to_numeric, errors='coerce')).round(2))

## À rapporter

1. Execution Accuracy est la métrique primaire.
2. Afficher les moyennes ± écarts-types des trois runs de la nouvelle étude complète.
3. Séparer explicitement les métriques BIRD et les ablations de retrieval sur TPC-DS / BIRD-Evidence-Corpus.
4. Documenter les refus, erreurs structurelles et erreurs sémantiques annotées.
5. Rapporter le test de McNemar pour H1 et l'accord inter-annotateur.

Pour régénérer la matrice et les figures :

```powershell
python src/build_comparative_matrix.py --results_dir results/pipelines_A_B_C_D --evaluation_dir data/evaluation --error_classification_dir results/error_classification --schema_ablation_file results/retrieval_ablation/schema_ablation.json --business_ablation_file results/retrieval_ablation/business_ablation.json --bird_ablation_file results/retrieval_ablation/bird_evidence_ablation.json --output_dir results/figures
python src/generate_figures.py --matrix results/figures/comparative_matrix.json --schema_ablation results/retrieval_ablation/schema_ablation.json --h1_paired results/hypothesis_H1/h1_test_result_paired.json --output_dir results/figures
```